# 11. Monte Carlo Season Simulation

Part of the `nuclearff` [User Tutorial](https://nolmacdonald.github.io/nuclearff/tutorial/index.html) notebook series -- every notebook shares one running `./demo` directory and depends on the ones before it having already run. See [README.md](README.md) for the full run order. Run `10_draft_and_playoff_visuals.ipynb` first.

A projection (`07_projections.ipynb`) is a single number -- a mean. `nuclearff.simulation.montecarlo` turns a mean and a spread into a full distribution of plausible season outcomes: a floor, a median, a ceiling, and the probability of clearing any threshold you care about. Mirrors the [Simulation](https://nolmacdonald.github.io/nuclearff/tutorial/11_simulation.html) docs page.

## Simulating a real player's season

`simulate_player_season` needs a mean and standard deviation of *weekly* fantasy points, plus an optional skew (positive skew models the real shape of a fantasy scorer's week -- occasional big games pulling the tail right, more often than a symmetric normal distribution would predict). Compute those from a real player's actual 2024 weekly output under this league's scoring:

In [1]:
import polars as pl
from scipy.stats import skew

from nuclearff.config import load_league_config
from nuclearff.config.models import SimulationConfig
from nuclearff.nflverse import load_weekly_skill_stats
from nuclearff.scoring import ScoringEngine
from nuclearff.simulation import simulate_player_season, summarize_distribution

league_cfg = load_league_config("./demo/configs/leagues/1367225133634191360.yaml")
engine = ScoringEngine(league_cfg.scoring)

weekly = engine.score_frame(load_weekly_skill_stats([2024]))
chase_weeks = weekly.filter(pl.col("player_display_name") == "Ja'Marr Chase")[
    "fantasy_points"
].to_numpy()

mean_ppg, sd_ppg, skew_val = (
    float(chase_weeks.mean()),
    float(chase_weeks.std()),
    float(skew(chase_weeks)),
)
print(
    f"games={len(chase_weeks)} mean={mean_ppg:.2f} sd={sd_ppg:.2f} skew={skew_val:.2f}"
)

ScoringEngine has no mapping for 48 nonzero scoring key(s); these will not be counted toward fantasy points: blk_kick=2.0, blk_kick_ret_yd=0.04, def_3_and_out=0.25, def_4_and_stop=0.5, def_pr_yd=0.04, def_st_ff=1.0, def_st_fum_rec=1.0, def_st_td=6.0, def_td=6.0, ff=1.0, fg_ret_yd=0.04, fgm_0_19=3.0, fgm_20_29=3.0, fgm_30_39=3.0, fgm_40_49=4.0, fgm_50p=5.0, fgmiss=-1.0, fgmiss_0_19=-1.0, fgmiss_20_29=-1.0, fum_rec=2.0, fum_rec_td=6.0, fum_ret_yd=0.04, int=2.0, int_ret_yd=0.04, kr_yd=0.04, pr_yd=0.04, pts_allow_0=5.0, pts_allow_14_20=1.0, pts_allow_1_6=4.0, pts_allow_28_34=-1.0, pts_allow_35p=-4.0, pts_allow_7_13=3.0, sack=1.0, safe=2.0, st_ff=1.0, st_fum_rec=1.0, st_td=6.0, tkl_loss=0.25, xpm=1.0, xpmiss=-1.0, yds_allow_0_100=5.0, yds_allow_100_199=3.0, yds_allow_200_299=2.0, yds_allow_350_399=-1.0, yds_allow_400_449=-3.0, yds_allow_450_499=-5.0, yds_allow_500_549=-6.0, yds_allow_550p=-7.0


games=17 mean=23.71 sd=12.48 skew=1.13


That's Ja'Marr Chase's real, full 2024 season under this league's actual 6-point-passing-TD, full-PPR rules: a meaningfully positive skew, well above a symmetric distribution's 0 -- his week-to-week scoring really does have a long right tail of huge games, not just a wide spread around the mean. Simulate 10,000 full seasons from those three numbers:

In [2]:
samples = simulate_player_season(
    mean_ppg, sd_ppg, skew_val, games=17, n_simulations=10000, seed=2026
)
samples.shape, round(samples.mean(), 1)

((10000,), np.float64(402.9))

`samples` is 10,000 simulated season totals -- one full 17-game season per row, resampled from the fitted skew-normal weekly distribution. The mean of 10,000 simulated seasons should closely track the simple `mean_ppg * 17` estimate, which is exactly what a well-calibrated simulation should do:

In [3]:
round(mean_ppg * 17, 1)

403.0

## Summarizing the distribution

Raw samples aren't useful on their own -- reduce them to the numbers you'd actually put in a report with `summarize_distribution`:

In [4]:
sim_cfg = SimulationConfig()  # floor=p10, ceiling=p90 by default
summary = summarize_distribution(samples, sim_cfg, top_n_threshold=330.0)
summary

{'floor': 338.729509155259,
 'median': 401.8525517897094,
 'ceiling': 469.26372300955785,
 'mean': 402.9307194002609,
 'p_exceeds_threshold': 0.9259}

Read against real 2024 WR1 fantasy-point totals, a 330-point threshold is a real "finished as a top-flight WR1" bar -- this simulation's `p_exceeds_threshold` is Chase's real probability of clearing it, given only his own real 2024 week-to-week volatility. `floor`/`ceiling` come from `SimulationConfig`'s configurable percentiles (10th/90th by default, from `02_configuration.ipynb`) -- not hardcoded, so a more conservative *or* more aggressive range is a config change, not a code change.

## Driving simulation from config

`simulate_from_config` is the same simulation, but reading its knobs (`games`, `n_simulations`) straight from a `SimulationConfig` instead of repeating them as keyword arguments -- the version you'd actually wire into a pipeline that already has a loaded project config:

In [5]:
from nuclearff.simulation import simulate_from_config

samples2 = simulate_from_config(mean_ppg, sd_ppg, skew_val, sim_cfg, seed=2026)
samples2.shape, round(samples2.mean(), 1)

((10000,), np.float64(402.9))

> **Why this matters:** a single mean projection and a full simulated distribution answer different questions, and it's worth being precise about which one you're actually asking. "Who is the best projected WR1" is a mean-projection question -- `07_projections.ipynb` answers it. "How much risk am I taking on by drafting this specific player instead of a safer one with a similar mean" is a distribution question -- only this chapter's `floor`/`ceiling` spread answers that, and two players with an identical mean can have very different floors.

## What's Next

A simulation is only as good as the model behind its mean and spread. `12_backtesting.ipynb` covers how to actually check that -- scoring a model against real past seasons it never saw, rather than trusting its numbers on faith.

## See Also

- `07_projections.ipynb` -- where the mean this chapter simulates around comes from.
- [Simulation](https://nolmacdonald.github.io/nuclearff/tutorial/11_simulation.html) -- the matching docs page.